In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
from src.utils import setup_logging
from src.optimization import PortfolioOptimizer
from src.financial_statistics import FinancialStatistics
from src.stress_test import PortfolioStressTester

In [3]:
# 1. Initialize Logging and Data Ingestion
logger = setup_logging()
logger.info("Executing Phase 7: Portfolio Stress Testing Notebook Pipeline.")

simple_returns = pd.read_csv("data/processed/simple_returns.csv", index_col="Date", parse_dates=True)

2026-07-13 00:12:19 [INFO] utils: Logging infrastructure successfully initialized.
2026-07-13 00:12:19 [INFO] 4097361403: Executing Phase 7: Portfolio Stress Testing Notebook Pipeline.


Writing logs to: /Users/admin/Desktop/Portofolio Analyzer/logs/portfolio_analyzer.log


In [4]:
# 2. Re-generate our two optimized portfolios to get weights
stats_engine = FinancialStatistics()
summary_stats = stats_engine.compute_summary_statistics(simple_returns, simple_returns)
matrices = stats_engine.get_matrices(simple_returns)

optimizer = PortfolioOptimizer(rf_rate=0.04)
max_sharpe_w = optimizer.optimize_max_sharpe(summary_stats["Annualized Return"], matrices["covariance"])
min_var_w = optimizer.optimize_min_variance(summary_stats["Annualized Return"], matrices["covariance"])


2026-07-13 00:12:37 [INFO] financial_statistics: Generating baseline financial statistics profiles...
2026-07-13 00:12:37 [INFO] financial_statistics: Computing asset covariance and correlation matrices.
2026-07-13 00:12:37 [INFO] optimization: Executing Max Sharpe Optimization solver.
2026-07-13 00:12:37 [INFO] optimization: Executing Minimum Variance Optimization solver.


In [5]:
# 3. Instantiate and Execute Stress Tester
tester = PortfolioStressTester()

print("\n=== STRESS TEST: MAXIMUM SHARPE PORTFOLIO ===")
max_sharpe_stress = tester.run_stress_test(simple_returns, max_sharpe_w)
display(max_sharpe_stress)

print("\n=== STRESS TEST: MINIMUM VARIANCE PORTFOLIO ===")
min_var_stress = tester.run_stress_test(simple_returns, min_var_w)
display(min_var_stress)

2026-07-13 00:12:55 [INFO] stress_test: Initiating historical portfolio stress testing sequence.



=== STRESS TEST: MAXIMUM SHARPE PORTFOLIO ===


,Start Date,End Date,Total Return,Max Drawdown
2020 COVID Crash,2020-02-19,2020-03-23,-13.72%,-15.89%
2022 Inflationary Bear Market,2022-01-03,2022-12-30,-11.58%,-20.15%
2024 Tech/Carry-Trade Shock,2024-07-16,2024-08-07,-5.80%,-6.81%


2026-07-13 00:12:55 [INFO] stress_test: Initiating historical portfolio stress testing sequence.



=== STRESS TEST: MINIMUM VARIANCE PORTFOLIO ===


,Start Date,End Date,Total Return,Max Drawdown
2020 COVID Crash,2020-02-19,2020-03-23,3.18%,-4.78%
2022 Inflationary Bear Market,2022-01-03,2022-12-30,-15.41%,-17.65%
2024 Tech/Carry-Trade Shock,2024-07-16,2024-08-07,1.01%,-1.34%
